In [ ]:
import os
import sys

current_dir = os.getcwd()
module_path = os.path.abspath(os.path.join(current_dir, '../..'))
if module_path not in sys.path:
    sys.path.insert(0, module_path)

In [2]:
import os
import sys

current_dir = os.getcwd()
module_path = os.path.abspath(os.path.join(current_dir, "../.."))
if module_path not in sys.path:
    sys.path.insert(0, module_path)
import os
import random
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy import stats
import pandas as pd

from dss import (
    SimpleNNClassification,
    train_model_to_threshold_classifier,
    compute_avg_loss,
    estimate_energy_gap_dss_pair,
    device,
)


def build_cancer_dataset(seed=42):
    data = load_breast_cancer()
    X = data.data
    y = data.target
    X_train, _, y_train, _ = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    return torch.utils.data.TensorDataset(X_train, y_train)


def make_loader(
    dataset,
    batch_size=32,
    shuffle=True,
    seed=None,
):
    if seed is None:
        worker_init = lambda wid: np.random.seed(wid)
    else:
        worker_init = lambda wid: np.random.seed(seed + wid)
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        worker_init_fn=worker_init,
    )


def make_subset_loader(
    dataset,
    subset_size,
    batch_size,
    seed,
):
    if subset_size is None or subset_size >= len(dataset):
        return make_loader(dataset, batch_size=batch_size, shuffle=True, seed=seed)
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(dataset), size=subset_size, replace=False)
    subset = torch.utils.data.Subset(dataset, idx)
    return make_loader(subset, batch_size=batch_size, shuffle=True, seed=seed)


def progress(iterable, desc=None, total=None, enabled=True):
    if not enabled:
        return iterable
    try:
        from tqdm import tqdm  # type: ignore
    except Exception:
        return iterable
    return tqdm(iterable, desc=desc, total=total)

def _save_checkpoint(path, payload):
    if not path:
        return
    tmp_path = f"{path}.tmp"
    torch.save(payload, tmp_path)
    os.replace(tmp_path, path)

def _load_checkpoint(path):
    if not path or not os.path.exists(path):
        return None
    try:
        return torch.load(path, map_location="cpu")
    except Exception as exc:
        print(f"Checkpoint load failed ({path}): {exc}")
        return None


def train_model_pool(
    cls,
    train_loader,
    loss_func,
    input_size,
    hidden_size,
    output_size,
    trainer,
    l1_lambda,
    num_models,
    max_epochs,
    seed_base,
    normalize_first_layer_weights=False,
    use_tqdm=False,
):
    params_list = []
    losses_total = []
    losses_data = []
    losses_l1 = []
    for i in progress(range(num_models), desc=f"pool w={hidden_size}", enabled=use_tqdm):
        torch.manual_seed(seed_base + i)
        model = cls(
            input_size=input_size, hidden_size=hidden_size, output_size=output_size
        ).to(device)
        trainer(
            model,
            train_loader,
            max_epochs=max_epochs,
            threshold=float("-inf"),
            l1_lambda=l1_lambda,
            normalize_first_layer_weights=normalize_first_layer_weights,
        )
        loss_total = float(compute_avg_loss(model, train_loader, loss_func, l1_lambda))
        loss_data = float(compute_avg_loss(model, train_loader, loss_func, 0.0))
        loss_l1 = loss_total - loss_data
        params = [p.detach().cpu().clone() for p in model.parameters()]
        params_list.append(params)
        losses_total.append(loss_total)
        losses_data.append(loss_data)
        losses_l1.append(loss_l1)
    return (
        params_list,
        np.asarray(losses_total, dtype=float),
        np.asarray(losses_data, dtype=float),
        np.asarray(losses_l1, dtype=float),
    )


def sample_pairs(indices, num_pairs, rng):
    pairs = []
    if len(indices) < 2:
        return pairs
    while len(pairs) < num_pairs:
        a, b = rng.choice(indices, size=2, replace=False)
        pairs.append((int(a), int(b)))
    return pairs


def cliffs_delta(b, a):
    b = np.asarray(b)
    a = np.asarray(a)
    gt = 0
    lt = 0
    for x in b:
        gt += np.sum(x > a)
        lt += np.sum(x < a)
    denom = len(a) * len(b)
    return (gt - lt) / denom if denom > 0 else np.nan


def bootstrap_diff(a, b, stat="mean", n_boot=2000, seed=0):
    a = np.asarray(a)
    b = np.asarray(b)
    if len(a) == 0 or len(b) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    if stat == "median":
        f = np.median
    elif stat == "max":
        f = np.max
    else:
        f = np.mean
    diffs = []
    for _ in range(n_boot):
        a_s = rng.choice(a, size=len(a), replace=True)
        b_s = rng.choice(b, size=len(b), replace=True)
        diffs.append(f(b_s) - f(a_s))
    diff = f(b) - f(a)
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    return float(diff), float(ci_low), float(ci_high)

def permutation_test_max(a, b, n_perm=2000, seed=0):
    a = np.asarray(a)
    b = np.asarray(b)
    if len(a) == 0 or len(b) == 0:
        return np.nan
    rng = np.random.default_rng(seed)
    observed = float(np.max(b) - np.max(a))
    combined = np.concatenate([a, b])
    n_a = len(a)
    diffs = []
    for _ in range(n_perm):
        rng.shuffle(combined)
        a_s = combined[:n_a]
        b_s = combined[n_a:]
        diffs.append(np.max(b_s) - np.max(a_s))
    diffs = np.asarray(diffs)
    p = float(np.mean(diffs <= observed))
    return p


In [3]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

dataset = build_cancer_dataset()

input_size = dataset.tensors[0].shape[1]
output_size = 1
l1_lambda = 0.001
loss_func = nn.functional.binary_cross_entropy_with_logits
batch_size = 10000

widths = [20, 200]
percentiles = [50, 10]
num_models = 1000
num_pairs = 200
max_epochs_pool = 150
max_epochs_dss = 150
max_depth = 8
path_steps = 200
dss_subset_size = None  # None for full dataset
normalize_first_layer_weights = True
use_tqdm = True
log_every_pairs = 5
ref_width = widths[0]
threshold_mode = "pairwise"  # "percentile" | "pairwise"

checkpoint_path = "energy_gap_cancer_checkpoint.pt"
checkpoint_every = 10
resume = True
save_partial_csv = True
out_csv = "energy_gap_cancer_percentiles.csv"

train_loader = make_loader(dataset, batch_size=batch_size, shuffle=True, seed=123)

config = {
    "widths": widths,
    "percentiles": percentiles,
    "num_models": num_models,
    "num_pairs": num_pairs,
    "max_epochs_pool": max_epochs_pool,
    "max_epochs_dss": max_epochs_dss,
    "max_depth": max_depth,
    "path_steps": path_steps,
    "dss_subset_size": dss_subset_size,
    "l1_lambda": l1_lambda,
    "batch_size": batch_size,
    "ref_width": ref_width,
    "input_size": input_size,
    "output_size": output_size,
    "normalize_first_layer_weights": normalize_first_layer_weights,
    "threshold_mode": threshold_mode,
}

state = _load_checkpoint(checkpoint_path) if resume else None
if state and state.get("config") != config:
    print("Checkpoint config mismatch. Ignoring checkpoint.")
    state = None

pools = state.get("pools", {}) if state else {}
rows = state.get("rows", []) if state else []
pairs_cache = state.get("pairs_cache", {}) if state else {}
start_task_index = int(state.get("task_index", 0)) if state else 0
if start_task_index > 0:
    print(f"Resuming from checkpoint at pair index {start_task_index}")

for hidden_size in widths:
    if hidden_size in pools:
        continue
    seed_base = 1000 + hidden_size
    params_list, losses_total, losses_data, losses_l1 = train_model_pool(
        SimpleNNClassification,
        train_loader,
        loss_func,
        input_size,
        hidden_size,
        output_size,
        train_model_to_threshold_classifier,
        l1_lambda,
        num_models,
        max_epochs_pool,
        seed_base,
        normalize_first_layer_weights=normalize_first_layer_weights,
        use_tqdm=use_tqdm,
    )
    pools[hidden_size] = (params_list, losses_total, losses_data, losses_l1)
    print(
        f"[pool] width={hidden_size} total_mean={losses_total.mean():.6f} "
        f"data_mean={losses_data.mean():.6f} l1_mean={losses_l1.mean():.6f}"
    )
    _save_checkpoint(
        checkpoint_path,
        {
            "config": config,
            "pools": pools,
            "rows": rows,
            "pairs_cache": pairs_cache,
            "task_index": start_task_index,
        },
    )

ref_losses = pools[ref_width][1]
thresholds = {perc: float(np.percentile(ref_losses, perc)) for perc in percentiles}

task_index = 0
for hidden_size in widths:
    seed_base = 2000 + hidden_size
    params_list, losses_total, losses_data, losses_l1 = pools[hidden_size]
    dss_loader = make_subset_loader(
        dataset, dss_subset_size, batch_size=batch_size, seed=seed_base + 777
    )

    perc_list = percentiles if threshold_mode == "percentile" else ["pairwise"]
    for perc in perc_list:
        if threshold_mode == "percentile":
            thr = thresholds[perc]
            eligible = np.where(losses_total <= thr)[0]
            print(
                f"[percentile] ref={ref_width} width={hidden_size} perc={perc} "
                f"threshold={thr:.6f} eligible={len(eligible)}"
            )
        else:
            thr = None
            eligible = np.arange(len(losses_total))
        pairs_key = f"{hidden_size}_{perc}"
        if pairs_key in pairs_cache:
            pairs = pairs_cache[pairs_key]
        else:
            if threshold_mode == "pairwise":
                rng = np.random.default_rng(seed_base + 7777)
            else:
                rng = np.random.default_rng(seed_base + int(perc * 10))
            pairs = sample_pairs(eligible, num_pairs, rng)
            pairs_cache[pairs_key] = pairs
        if len(pairs) < num_pairs:
            print(f"width={hidden_size} perc={perc}: not enough eligible points")

        for pair_idx, (i, j) in progress(
            list(enumerate(pairs)),
            desc=f"pairs w={hidden_size} p={perc}",
            enabled=use_tqdm,
        ):
            if task_index < start_task_index:
                task_index += 1
                continue
            if threshold_mode == "pairwise":
                thr = float(max(losses_total[i], losses_total[j]))
            gap, barrier, _, hit, _ = estimate_energy_gap_dss_pair(
                cls=SimpleNNClassification,
                loss_func=loss_func,
                trainer=train_model_to_threshold_classifier,
                theta_a=params_list[i],
                theta_b=params_list[j],
                dataloader=dss_loader,
                threshold=thr,
                max_depth=max_depth,
            l1_lambda=l1_lambda,
            max_epochs=max_epochs_dss,
            path_steps=path_steps,
            normalize_first_layer_weights=normalize_first_layer_weights,
            input_size=input_size,
            hidden_size=hidden_size,
            output_size=output_size,
        )
            rows.append(
                {
                    "width": hidden_size,
                    "percentile": perc,
                    "threshold": thr,
                    "pair": pair_idx,
                    "gap": gap,
                    "barrier": barrier,
                    "hit_max_depth": bool(hit),
                    "loss_a_total": float(losses_total[i]),
                    "loss_b_total": float(losses_total[j]),
                    "loss_a_data": float(losses_data[i]),
                    "loss_b_data": float(losses_data[j]),
                    "loss_a_l1": float(losses_l1[i]),
                    "loss_b_l1": float(losses_l1[j]),
                }
            )
            task_index += 1
            if checkpoint_every > 0:
                is_due = (task_index % checkpoint_every == 0)
                if is_due:
                    _save_checkpoint(
                        checkpoint_path,
                        {
                            "config": config,
                            "pools": pools,
                            "rows": rows,
                            "pairs_cache": pairs_cache,
                            "task_index": task_index,
                        },
                    )
                    if save_partial_csv:
                        pd.DataFrame(rows).to_csv(out_csv, index=False)
            if pair_idx % log_every_pairs == 0:
                print(
                    f"[gap] width={hidden_size} perc={perc} "
                    f"pair={pair_idx} gap={gap:.6f}"
                )

df = pd.DataFrame(rows)
df.to_csv(out_csv, index=False)
print(f"Saved {out_csv}")

summary = (
    df.groupby(["width", "percentile"])
    .agg(
        gap_mean=("gap", "mean"),
        gap_median=("gap", "median"),
        gap_max=("gap", "max"),
        hit_rate=("hit_max_depth", "mean"),
        n_pairs=("gap", "size"),
    )
    .reset_index()
)
print(summary)

if len(widths) == 2:
    w0, w1 = widths
    if threshold_mode == "pairwise":
        a = df[df["width"] == w0]["gap"].to_numpy()
        b = df[df["width"] == w1]["gap"].to_numpy()
        if len(a) > 0 and len(b) > 0:
            stat, p = stats.mannwhitneyu(b, a, alternative="less")
            delta = cliffs_delta(b, a)
            mean_diff, ci_l, ci_h = bootstrap_diff(a, b, stat="mean", n_boot=2000, seed=0)
            med_diff, mci_l, mci_h = bootstrap_diff(a, b, stat="median", n_boot=2000, seed=100)
            max_diff, max_ci_l, max_ci_h = bootstrap_diff(a, b, stat="max", n_boot=2000, seed=200)
            p_max = permutation_test_max(a, b, n_perm=2000, seed=300)
            print(
                f"pairwise: Mann-Whitney p={p:.6g} (H1: {w1} < {w0})\n"
                f"  mean diff(b-a)={mean_diff:.6f} CI=[{ci_l:.6f}, {ci_h:.6f}]\n"
                f"  median diff(b-a)={med_diff:.6f} CI=[{mci_l:.6f}, {mci_h:.6f}]\n"
                f"  max diff(b-a)={max_diff:.6f} CI=[{max_ci_l:.6f}, {max_ci_h:.6f}] p_perm={p_max:.6g}\n"
                f"  Cliff's delta={delta:.3f}"
            )
    else:
        for perc in percentiles:
            a = df[(df["width"] == w0) & (df["percentile"] == perc)]["gap"].to_numpy()
            b = df[(df["width"] == w1) & (df["percentile"] == perc)]["gap"].to_numpy()
            if len(a) == 0 or len(b) == 0:
                continue
            stat, p = stats.mannwhitneyu(b, a, alternative="less")
            delta = cliffs_delta(b, a)
            mean_diff, ci_l, ci_h = bootstrap_diff(a, b, stat="mean", n_boot=2000, seed=perc)
            med_diff, mci_l, mci_h = bootstrap_diff(a, b, stat="median", n_boot=2000, seed=perc + 100)
            max_diff, max_ci_l, max_ci_h = bootstrap_diff(a, b, stat="max", n_boot=2000, seed=perc + 200)
            p_max = permutation_test_max(a, b, n_perm=2000, seed=perc + 300)
            print(
                f"percentile={perc}: Mann-Whitney p={p:.6g} (H1: {w1} < {w0})\n"
                f"  mean diff(b-a)={mean_diff:.6f} CI=[{ci_l:.6f}, {ci_h:.6f}]\n"
                f"  median diff(b-a)={med_diff:.6f} CI=[{mci_l:.6f}, {mci_h:.6f}]\n"
                f"  max diff(b-a)={max_diff:.6f} CI=[{max_ci_l:.6f}, {max_ci_h:.6f}] p_perm={p_max:.6g}\n"
                f"  Cliff's delta={delta:.3f}"
            )


Checkpoint load failed (energy_gap_cancer_checkpoint.pt): Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray._reconstruct was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray._reconstruct])` or the `torch.serialization.safe_globals([numpy._core.multiarray._reconstruct])` context manager to allowlist this global if you trust this class/functio

pool w=20: 100%|███████████████████████████████████████████████████████████████████| 1000/1000 [06:28<00:00,  2.57it/s]


[pool] width=20 total_mean=0.028276 data_mean=0.006798 l1_mean=0.021478


pool w=200: 100%|██████████████████████████████████████████████████████████████████| 1000/1000 [06:30<00:00,  2.56it/s]


[pool] width=200 total_mean=0.032448 data_mean=0.002398 l1_mean=0.030049


...

[gap] width=200 perc=pairwise pair=190 gap=0.000000


pairs w=200 p=pairwise:  98%|██████████████████████████████████████████████████████▉ | 196/200 [05:34<00:05,  1.43s/it]

[gap] width=200 perc=pairwise pair=195 gap=0.000000


pairs w=200 p=pairwise: 100%|████████████████████████████████████████████████████████| 200/200 [05:42<00:00,  1.71s/it]

Saved energy_gap_cancer_percentiles.csv
   width percentile  gap_mean    gap_median   gap_max  hit_rate  n_pairs
0     20   pairwise  0.000793  1.177192e-09  0.010696     0.620      200
1    200   pairwise  0.000124  0.000000e+00  0.001477     0.415      200
pairwise: Mann-Whitney p=2.25842e-05 (H1: 200 < 20)
  mean diff(b-a)=-0.000669 CI=[-0.000985, -0.000374]
  median diff(b-a)=-0.000000 CI=[-0.000011, -0.000000]
  max diff(b-a)=-0.009219 CI=[-0.009614, -0.006470] p_perm=0
  Cliff's delta=-0.222
